# Silver

In [0]:
CATALOG = "crop_risk"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pyspark.sql.functions as F

Delta table info for `crop_trend_master`

In [0]:
bronze_crop_trend = spark.table(f"{CATALOG}.bronze.crop_trend_master")
bronze_crop_trend.show(10)

In [0]:
display(bronze_crop_trend.printSchema())

Duplicated values for `crop_trend_master`

In [0]:
key_cols = ["province", "crop", "year", "quarter"]
measure_col = "production"
audit_cols = ["_ingested_at", "_source_file"]

df_duplicates = (
    bronze_crop_trend.groupBy(*key_cols)
    .count()
    .withColumnRenamed("count", "production_count")
    .filter(F.col("count") > 1)
)

df_duplicates.show(10)

In [0]:
other_cols = [c for c in bronze_crop_trend.columns if c not in key_cols + [measure_col] + audit_cols]
 
bronze_crop_trend = (
    bronze_crop_trend.groupBy(*key_cols)
    .agg(
        F.sum(measure_col).alias(measure_col),
        *[F.first(c, ignorenulls=True).alias(c) for c in other_cols],
    )
)

print(f"crop_trend_master: {bronze_crop_trend.count():,} rows")

Missing values for `crop_trend_master`

In [0]:
bronze_crop_trend.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_cols + [measure_col]
]).show()

EDA

In [0]:
bronze_crop_trend.select("province").distinct().show(10)

In [0]:
bronze_crop_trend.select("crop").distinct().show(10)

In [0]:
bronze_crop_trend.select("crop_group").distinct().show(10)

In [0]:
bronze_crop_trend.select("risk_label").distinct().show(10)

In [0]:
bronze_crop_trend = bronze_crop_trend.withColumn("province", F.initcap(F.col("province")))
bronze_crop_trend.show(10)

In [0]:
"""

(
    df_crop_trend.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.crop_trend_master")
)

"""